# 03b — Views para o Random Forest binário COM os atributos de AUC alto (sem ablação)

> **Variante COM os atributos de AUC alto.** É a mesma preparação do notebook 03 (mesmas medianas do treino, mesma regra de nulos e de sobreposição), **sem retirar** `bwd_packets_s`, `inbound` e `total_backward_packets`. Serve para comparar o RF **com** e **sem** esses atributos, nas **mesmas linhas**. Todos os objetos criados aqui têm o sufixo `_com_auc`: **nada** da versão sem esses atributos é alterado.

Este notebook **não treina nada**. Ele cria duas **views** no ClickHouse, prontas para ler e treinar o RF binário, aplicando duas decisões do TCC (e **não** aplicando a ablação):

| Decisão | O que é feito nas views |
|---|---|
| **Valores nulos** | Substituídos pela **mediana do treino**, igual ao `PreProc.transformar(df, "rf")` do notebook 02. As colunas `<feature>_ausente` (1 = o valor estava vazio) continuam como atributos |
| **Sem ablação** (a diferença para o notebook 03) | Ficam **todas** as colunas da seleção binária, inclusive `bwd_packets_s`, `inbound` e `total_backward_packets` (e os indicadores `_ausente` delas, se existirem) |
| **Duplicatas treino × teste (opção A)** | O treino fica **completo**. O teste só tem as linhas com `sobrepoe_treino = 0`, ou seja, **sem** as linhas cujas características são idênticas às de alguma linha do treino |

**Views criadas** (nada existente é alterado; só são criadas views e uma tabela de auditoria com prefixo `prep_`):

| Objeto | Conteúdo |
|---|---|
| `prep_vw_rf_bin_treino_com_auc` | `id_linha`, `y` (0 = BENIGN, 1 = ataque), atributos sem nulos e indicadores `_ausente` |
| `prep_vw_rf_bin_teste_sem_sobrep_com_auc` | Igual, só com as linhas do teste **sem sobreposição** com o treino |
| `prep_views_rf_binario_com_auc` | Auditoria: cada atributo da seleção binária, se foi usado ou removido, e a mediana usada |

**Como ler no Python** (a coluna `y` é a resposta; `id_linha` **não** entra na entrada do modelo):
```python
tr = ler_rf_bin("treino", mod=20, resto=0)        # 1/20 do treino, determinístico
X, y = tr.drop(columns=["id_linha", "y"]), tr["y"]
```

**Antes de rodar:** o notebook 02 (v8.10 ou mais novo) precisa ter terminado. Aqui são lidas as tabelas `prep_final_treino`, `prep_final_teste`, `prep_parametros` e `prep_selecao_atributos`.

**Conexão:** o mesmo código do notebook 02, com as mesmas variáveis. Coloque este arquivo **na mesma pasta do notebook 02** (onde está o arquivo `.env`), que precisa ter `CLICKHOUSE_HOST`, `CLICKHOUSE_PORT`, `CLICKHOUSE_USER`, `CLICKHOUSE_PASSWORD` e `CLICKHOUSE_DATABASE` (e, opcionalmente, `CLICKHOUSE_SCHEME`, padrão `https`). A trava de segurança é a mesma: só é possível escrever em tabelas e views `prep_*`.

In [8]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse-production-e204.up.railway.app:443/railway  |  usuário: clickhouse


In [9]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [10]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [11]:
# ---------- ferramentas mínimas ----------
import numpy as np
from datetime import datetime

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def blocos_de(lista, n=20):
    return [lista[i:i + n] for i in range(0, len(lista), n)]

## 1. O que entra e o que sai (decisões desta etapa)

A célula abaixo lê do notebook 02: a **seleção binária** de atributos, as **medianas do treino** (`prep_parametros`) e quais atributos têm indicador `_ausente`. Depois monta a lista final de atributos:

**atributos finais = seleção binária completa (sem ablação)**

- `ATALHOS_ABLACAO` fica **vazia** de propósito: esta é a variante **COM** os atributos de AUC alto. Se você a preencher, a variante deixa de ser a "completa".
- A célula mostra os atalhos que o notebook 02 marcou (`atalho_suspeito = 1`): eles **continuam** nesta versão.
- Os indicadores `_ausente` das colunas de AUC alto (se houver) **voltam** junto com elas.

In [12]:
# ---------- 1.1 Parâmetros e leitura do que o notebook 02 gravou ----------
ATALHOS_ABLACAO = globals().get("ATALHOS_ABLACAO", [])   # variante COM os atributos de AUC alto: nenhuma coluna é retirada
VIEW_TREINO, VIEW_TESTE, TAB_AUD = "prep_vw_rf_bin_treino_com_auc", "prep_vw_rf_bin_teste_sem_sobrep_com_auc", "prep_views_rf_binario_com_auc"

faltam = [t for t in ("prep_final_treino", "prep_final_teste", "prep_parametros", "prep_selecao_atributos") if not existe(t)]
assert not faltam, f"Rode antes o notebook de pré-processamento (02): faltam as tabelas {faltam}"

SEL_BIN = consultar("SELECT feature FROM prep_selecao_atributos WHERE tarefa = 'binario' AND decisao = 'mantida' ORDER BY posicao")["feature"].tolist()
par = consultar_num("SELECT coluna, mediana FROM prep_parametros WHERE usa_indicador = 1")
MEDIANA = dict(zip(par.coluna, par.mediana))
cols_final = consultar("SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = 'prep_final_treino'")["name"].tolist()
COM_IND = {c[:-len("_ausente")] for c in cols_final if c.endswith("_ausente")}
assert set(MEDIANA) == COM_IND, f"as colunas com mediana em prep_parametros ({sorted(MEDIANA)}) não coincidem com as colunas *_ausente de prep_final_treino ({sorted(COM_IND)})"
assert not any(pd.isna(v) for v in MEDIANA.values()), "há coluna com indicador mas sem mediana em prep_parametros"

marcadas = consultar("SELECT feature FROM prep_selecao_atributos WHERE tarefa = 'binario' AND decisao = 'mantida' AND atalho_suspeito = 1 ORDER BY posicao")["feature"].tolist()
print("Seleção binária do notebook 02:", len(SEL_BIN), "atributos")
print("Atalhos que VOCÊ definiu para a ablação:      ", ATALHOS_ABLACAO)
print("Atalhos marcados pelo notebook 02 (binário):  ", marcadas)
print("⚠ ATALHOS_ABLACAO não está vazia: esta variante deixou de ser a versão COM todos os atributos." if ATALHOS_ABLACAO else "✔ Variante COM os atributos de AUC alto: nenhuma coluna será retirada (os atalhos marcados pelo notebook 02 continuam na seleção).")
fora = [c for c in ATALHOS_ABLACAO if c not in SEL_BIN]
if fora: print("⚠ não estão na seleção binária e serão ignorados:", fora)
RETIRADOS = [c for c in SEL_BIN if c in set(ATALHOS_ABLACAO)]
FEATS = [c for c in SEL_BIN if c not in set(ATALHOS_ABLACAO)]
IND = [c for c in FEATS if c in MEDIANA]
assert FEATS, "nenhum atributo restou depois da ablação"
print(f"\nRetirados pela ablação ({len(RETIRADOS)}): {RETIRADOS}")
print(f"Atributos finais: {len(FEATS)} (+ {len(IND)} indicadores _ausente: {[c + '_ausente' for c in IND]})")
print("Medianas do treino usadas na imputação:", {c: MEDIANA[c] for c in IND})

Seleção binária do notebook 02: 45 atributos
Atalhos que VOCÊ definiu para a ablação:       []
Atalhos marcados pelo notebook 02 (binário):   ['bwd_packets_s', 'inbound', 'total_backward_packets']
✔ Variante COM os atributos de AUC alto: nenhuma coluna será retirada (os atalhos marcados pelo notebook 02 continuam na seleção).

Retirados pela ablação (0): []
Atributos finais: 45 (+ 1 indicadores _ausente: ['flow_bytes_s_ausente'])
Medianas do treino usadas na imputação: {'flow_bytes_s': 689.5911165696316}


## 2. Criação das views

**Valores nulos.** Nas colunas que têm ausentes, a view troca `NULL` pela **mediana do treino**: `ifNull(coluna, mediana)`. O número da mediana vem de `prep_parametros` e fica **escrito dentro da view**, então a view sempre devolve os mesmos valores. A mediana é do **treino** para os dois conjuntos (o teste não calcula a dele), como no `PreProc`.

**Sem ablação.** Todas as colunas da seleção binária aparecem na view.

**Opção A.** A view de teste filtra `sobrepoe_treino = 0`. A view de treino não filtra nada.

**Se o notebook 02 for refeito** e as medianas mudarem, **rode este notebook de novo** para recriar as views com os valores novos.

In [13]:
# ---------- 2.1 Views definitivas e tabela de auditoria ----------
versao_views = datetime.now().strftime("%Y%m%d_%H%M%S")
def _expr(c):
    return f"ifNull(toFloat64({q(c)}), {float(MEDIANA[c])!r}) AS {q(c)}" if c in MEDIANA else f"toFloat64({q(c)}) AS {q(c)}"
sel_cols = ", ".join([_expr(c) for c in FEATS] + [f"{q(c + '_ausente')} AS {q(c + '_ausente')}" for c in IND])

executar(f"DROP VIEW IF EXISTS `{VIEW_TREINO}`")
executar(f"CREATE VIEW {VIEW_TREINO} AS SELECT id_linha, label_binary AS y, {sel_cols} FROM prep_final_treino")
executar(f"DROP VIEW IF EXISTS `{VIEW_TESTE}`")
executar(f"CREATE VIEW {VIEW_TESTE} AS SELECT id_linha, label_binary AS y, {sel_cols} FROM prep_final_teste WHERE sobrepoe_treino = 0")

executar(f"DROP TABLE IF EXISTS `{TAB_AUD}`")
executar(f"CREATE TABLE {TAB_AUD} (posicao UInt16, feature String, situacao String, motivo String, mediana_imputacao Nullable(Float64), tem_indicador UInt8, versao String) ENGINE = MergeTree ORDER BY posicao")
linhas = []
for pos, c in enumerate(SEL_BIN, 1):
    usada = c not in set(ATALHOS_ABLACAO)
    med = f"{float(MEDIANA[c])!r}" if (usada and c in MEDIANA) else "NULL"
    linhas.append(f"({pos}, {lit(c)}, {lit('usada' if usada else 'removida')}, {lit('' if usada else 'ablação: AUC alto (atalho suspeito)')}, {med}, {int(usada and c in MEDIANA)}, {lit(versao_views)})")
executar(f"INSERT INTO {TAB_AUD} VALUES " + ", ".join(linhas))
print(f"✔ Views criadas: {VIEW_TREINO} e {VIEW_TESTE}\n✔ Auditoria: {TAB_AUD} ({len(linhas)} atributos)  | versão {versao_views}")
print("\nDefinição gravada da view de teste:\n" + str(consultar(f"SHOW CREATE VIEW {VIEW_TESTE}").iloc[0, 0])[:700].replace("\\n", "\n") + " ...")

✔ Views criadas: prep_vw_rf_bin_treino_com_auc e prep_vw_rf_bin_teste_sem_sobrep_com_auc
✔ Auditoria: prep_views_rf_binario_com_auc (45 atributos)  | versão 20261024_001223

Definição gravada da view de teste:
CREATE VIEW railway.prep_vw_rf_bin_teste_sem_sobrep_com_auc
(
    `id_linha` UInt64,
    `y` UInt8,
    `bwd_packets_s` Float64,
    `avg_fwd_segment_size` Float64,
    `inbound` Float64,
    `bwd_header_length` Float64,
    `total_backward_packets` Float64,
    `max_packet_length` Float64,
    `bwd_iat_max` Float64,
    `bwd_iat_total` Float64,
    `bwd_iat_mean` Float64,
    `fwd_packet_length_max` Float64,
    `total_length_of_fwd_packets` Float64,
    `bwd_iat_min` Float64,
    `down_up_ratio` Float64,
    `flow_bytes_s` Float64,
    `flow_iat_max` Float64,
    `packet_length_std` Float64,
    `packet_length_variance` Float64,
    `bwd_packet_length_max` Float64,
    `total_length_of_bwd ...


## 3. Validação

Confere, de forma independente, que as views fazem o que foi decidido:

1. as colunas são exatamente as esperadas, e nenhum rótulo, `origem_tabela` ou marca de controle aparece;
2. o treino tem **todas** as linhas de `prep_final_treino`, e o teste tem as linhas **sem** sobreposição;
3. `y` só tem 0 e 1, e as duas classes existem nos dois conjuntos;
4. **não há nulo nem infinito** em nenhum atributo;
5. **equivalência com o `PreProc`:** numa amostra, o resultado da view é **igual** ao que se obtém lendo as linhas originais e aplicando a imputação do RF do `PreProc` (completar com a mediana do treino e criar os indicadores).

In [14]:
# ---------- 3.1 Validação ----------
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)

esperadas = ["id_linha", "y"] + FEATS + [c + "_ausente" for c in IND]
proibidas = {"label_original", "label_multi", "label_binary", "origem_tabela", "n_repeticoes", "conflito_rotulo", "conflito_binario", "sobrepoe_treino", "sobrepoe_treino_mesmo_rotulo", "b"} | set(RETIRADOS) | {c + "_ausente" for c in RETIRADOS}
for v in (VIEW_TREINO, VIEW_TESTE):
    cols_v = consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(v)} ORDER BY position")["name"].tolist()
    verifica(f"{v}: colunas = id_linha, y, atributos finais e seus indicadores", cols_v == esperadas, f"({len(cols_v)} colunas)")
    verifica(f"{v}: sem rótulo, marcas de controle nem colunas retiradas na ablação", not (proibidas & set(cols_v)))

n_tr, n_te, n_sem = n_linhas("prep_final_treino"), n_linhas("prep_final_teste"), n_linhas("prep_final_teste", "sobrepoe_treino = 0")
verifica("Treino: todas as linhas de prep_final_treino", n_linhas(VIEW_TREINO) == n_tr, f"({n_tr:,})")
verifica("Teste: só as linhas sem sobreposição com o treino", n_linhas(VIEW_TESTE) == n_sem, f"({n_sem:,} de {n_te:,}; {n_te - n_sem:,} removidas)")
for v in (VIEW_TREINO, VIEW_TESTE):
    d = consultar_num(f"SELECT y, count() AS n FROM {v} GROUP BY y")
    verifica(f"{v}: y só com 0 e 1, e as duas classes presentes", set(d.y.astype(int)) == {0, 1}, f"({dict(zip(d.y.astype(int), d.n.astype(int)))})")

colunas_feat = FEATS
for v in (VIEW_TREINO, VIEW_TESTE):
    ruins = {}
    for b in blocos_de(colunas_feat, 20):
        r = consultar_num(f"SELECT " + ", ".join(f"countIf(isNull({q(c)}) OR isInfinite({q(c)})) AS {q(c)}" for c in b) + f" FROM {v}").iloc[0]
        ruins.update({c: int(n) for c, n in r.items() if n > 0})
    verifica(f"{v}: nenhum nulo nem infinito nos atributos", not ruins, f"({ruins})" if ruins else "")

def _equivale(view, fonte, extra, mod=1500):
    # Reaplica, em pandas, a mesma regra do PreProc.transformar(..., 'rf') às linhas originais e compara com a view.
    # A amostra tem linhas sorteadas (1/mod) MAIS linhas com valor ausente (1/100 delas, para caber na memória), para a imputação ser de fato exercitada.
    ausente = " OR ".join(f"isNull({q(c)})" for c in IND)
    cond = f"(cityHash64(id_linha) % {mod} = 0" + (f" OR (({ausente}) AND cityHash64(id_linha) % 100 = 1)" if IND else "") + f"){extra}"
    cols = FEATS + [c + "_ausente" for c in IND]
    v = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in cols) + f" FROM {view} WHERE id_linha IN (SELECT id_linha FROM {fonte} WHERE {cond}) ORDER BY id")
    r = consultar(f"SELECT toString(id_linha) AS id, label_binary AS y, " + ", ".join(q(c) for c in FEATS) + f" FROM {fonte} WHERE {cond} ORDER BY id")
    if len(v) != len(r) or (v["id"].values != r["id"].values).any(): return len(v), 0, False
    X = r[FEATS].apply(pd.to_numeric, errors="coerce").astype("float64").replace([np.inf, -np.inf], np.nan)
    ind = {c + "_ausente": X[c].isna().astype("uint8") for c in IND}
    n_aus = int(pd.DataFrame(ind, index=X.index).max(axis=1).sum()) if IND else 0
    for c in IND: X[c] = X[c].fillna(MEDIANA[c])
    esp = pd.concat([X, pd.DataFrame(ind, index=X.index)], axis=1)
    obt = v[list(esp.columns)].apply(pd.to_numeric, errors="coerce")
    return len(v), n_aus, bool(np.allclose(obt.values.astype(float), esp.values.astype(float), rtol=1e-12, atol=1e-12) and (v["y"].astype(int).values == r["y"].astype(int).values).all())
for v, fonte, extra, nome in ((VIEW_TREINO, "prep_final_treino", "", "treino"), (VIEW_TESTE, "prep_final_teste", " AND sobrepoe_treino = 0", "teste")):
    n_am, n_aus, igual = _equivale(v, fonte, extra)
    verifica(f"Equivalência com o PreProc (RF) em amostra do {nome}", igual and n_am > 0, f"({n_am:,} linhas comparadas, {n_aus:,} delas com valor ausente imputado)")
    if IND and n_aus == 0: print(f"   ⚠ a amostra do {nome} não teve nenhuma linha com valor ausente: a imputação não foi exercitada nesta conferência.")
verifica("Auditoria: uma linha por atributo da seleção binária", n_linhas(TAB_AUD) == len(SEL_BIN), f"({n_linhas(TAB_AUD)})")
colunas_view = lambda vw: consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(vw)} ORDER BY position")["name"].tolist()
if existe("prep_vw_rf_bin_treino") and existe("prep_vw_rf_bin_teste_sem_sobrep"):
    for v_com, v_sem in ((VIEW_TREINO, "prep_vw_rf_bin_treino"), (VIEW_TESTE, "prep_vw_rf_bin_teste_sem_sobrep")):
        cs, cc = colunas_view(v_sem), colunas_view(v_com); comuns = [c for c in cs if c not in ("id_linha", "y")]; extras = [c for c in cc if c not in cs]
        verifica(f"{v_com}: contém todas as colunas de {v_sem} e só acrescenta colunas de AUC alto (e seus indicadores)",
                 set(cs) <= set(cc) and len(extras) > 0 and all((c in marcadas) or (c.endswith("_ausente") and c[:-8] in marcadas) for c in extras), f"(acrescenta: {extras})")
        a_ = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in comuns) + f" FROM {v_sem} WHERE cityHash64(id_linha) % 300 = 0 ORDER BY id")
        b_ = consultar(f"SELECT toString(id_linha) AS id, y, " + ", ".join(q(c) for c in comuns) + f" FROM {v_com} WHERE cityHash64(id_linha) % 300 = 0 ORDER BY id")
        verifica(f"{v_com}: valores das colunas comuns idênticos aos de {v_sem} (amostra)", len(a_) == len(b_) > 0 and (a_["id"].values == b_["id"].values).all()
                 and np.allclose(a_.drop(columns="id").apply(pd.to_numeric).values.astype(float), b_.drop(columns="id").apply(pd.to_numeric).values.astype(float), rtol=1e-12, atol=1e-12), f"({len(a_):,} linhas)")
else:
    print("(as views do notebook 03 não existem: a comparação com a versão sem os atributos de AUC alto fica para depois)")
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas.")

✔ prep_vw_rf_bin_treino_com_auc: colunas = id_linha, y, atributos finais e seus indicadores (48 colunas)
✔ prep_vw_rf_bin_treino_com_auc: sem rótulo, marcas de controle nem colunas retiradas na ablação 
✔ prep_vw_rf_bin_teste_sem_sobrep_com_auc: colunas = id_linha, y, atributos finais e seus indicadores (48 colunas)
✔ prep_vw_rf_bin_teste_sem_sobrep_com_auc: sem rótulo, marcas de controle nem colunas retiradas na ablação 
✔ Treino: todas as linhas de prep_final_treino (8,257,343)
✔ Teste: só as linhas sem sobreposição com o treino (7,219,189 de 20,364,525; 13,145,336 removidas)
✔ prep_vw_rf_bin_treino_com_auc: y só com 0 e 1, e as duas classes presentes ({0: 50768, 1: 8206575})
✔ prep_vw_rf_bin_teste_sem_sobrep_com_auc: y só com 0 e 1, e as duas classes presentes ({0: 56216, 1: 7162973})
✔ prep_vw_rf_bin_treino_com_auc: nenhum nulo nem infinito nos atributos 
  conexão instável (Exception: HTTP 500 [código ClickHouse: 241] corpo='{\n\t"meta":\n\t[\n\t\t{\n\t\t\t"name": "bwd_packets_s",

## 4. Resumo e leitura em lotes

Mostra o tamanho de cada conjunto e, principalmente, **quantas linhas de cada classe a opção A tirou do teste**. Se sobrarem poucas linhas BENIGN no teste sem sobreposição, as métricas do BENIGN ficam menos estáveis, e isso precisa aparecer no TCC.

A função `ler_rf_bin` lê as views em lotes (`mod` e `resto`), porque o treino tem milhões de linhas e não deve ser carregado de uma vez. Cada lote é a fração `resto/mod` das linhas, sorteada de forma **determinística** por `id_linha`.

In [15]:
# ---------- 4.1 Resumo e leitura em lotes ----------
d = consultar_num(f"SELECT 'treino' AS conjunto, y, count() AS linhas FROM {VIEW_TREINO} GROUP BY y "
                  f"UNION ALL SELECT 'teste (sem sobreposição)', y, count() FROM {VIEW_TESTE} GROUP BY y "
                  f"UNION ALL SELECT 'teste (completo)', label_binary, count() FROM prep_final_teste GROUP BY label_binary")
t = d.pivot_table(index="y", columns="conjunto", values="linhas", aggfunc="sum", fill_value=0).astype(int)
t = t[["treino", "teste (completo)", "teste (sem sobreposição)"]]
t["removidas pela opção A"] = t["teste (completo)"] - t["teste (sem sobreposição)"]
t["% removido"] = (100 * t["removidas pela opção A"] / t["teste (completo)"]).round(1)
t.index = t.index.map({0: "0 (BENIGN)", 1: "1 (ataque)"})
print("== Linhas por classe =="); display(t)
nb_te = int(t.loc["0 (BENIGN)", "teste (sem sobreposição)"])
print(f"\nBENIGN no teste sem sobreposição: {nb_te:,}" + ("  ⚠ poucas linhas: as métricas do BENIGN ficam instáveis; relate isso." if nb_te < 1000 else ""))
print(f"Atributos: {len(FEATS)} + {len(IND)} indicadores = {len(FEATS) + len(IND)} colunas de entrada. RAM estimada p/ o treino inteiro em float64: {n_tr * (len(FEATS) + len(IND)) * 8 / 2**30:.1f} GiB (por isso, ler em lotes).")

def ler_rf_bin(conjunto, mod=None, resto=0, limite=None):
    # conjunto: 'treino' ou 'teste'. Lê a fração resto/mod das linhas (determinística); sem mod, lê tudo (cuidado com o treino).
    view = VIEW_TREINO if conjunto == "treino" else VIEW_TESTE
    onde = f" WHERE cityHash64(id_linha) % {int(mod)} = {int(resto)}" if mod else ""
    return consultar_num(f"SELECT * FROM {view}{onde}" + (f" LIMIT {int(limite)}" if limite else ""))

for conj in ("treino", "teste"):                                   # teste rápido de leitura (1/500 de cada conjunto)
    a = ler_rf_bin(conj, mod=500, resto=0)
    assert not a.drop(columns=["id_linha"]).isna().any().any(), "NaN na leitura"
    print(f"Leitura de teste ({conj}, 1/500): {a.shape[0]:,} linhas x {a.shape[1]} colunas, sem NaN | classes: {a['y'].value_counts().to_dict()}")

== Linhas por classe ==


conjunto,treino,teste (completo),teste (sem sobreposição),removidas pela opção A,% removido
y,,,,,
0 (BENIGN),50768,56965,56216,749,1.3
1 (ataque),8206575,20307560,7162973,13144587,64.7



BENIGN no teste sem sobreposição: 56,216
Atributos: 45 + 1 indicadores = 46 colunas de entrada. RAM estimada p/ o treino inteiro em float64: 2.8 GiB (por isso, ler em lotes).
Leitura de teste (treino, 1/500): 16,812 linhas x 48 colunas, sem NaN | classes: {1: 16702, 0: 110}
Leitura de teste (teste, 1/500): 14,268 linhas x 48 colunas, sem NaN | classes: {1: 14159, 0: 109}
